In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import natural_units as nu
from scipy.special import erf
import scipy.integrate
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
from scipy.interpolate import interp1d
import re
# multi-core/thread:
import concurrent.futures

In [2]:
# First of all, define the mass and cross section grid we are working with.
log_m_min  = -3
log_m_max  = 1
n_m    = 17   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
log_cs_min = -3
log_cs_max = 0
n_cs   = 25
m_grid  = np.logspace(log_m_min, log_m_max, n_m)
cs_grid = np.logspace(log_cs_min, log_cs_max, n_cs)
center_line = np.array([2.15504637e-23, 1.53030461e-23, 1.26359147e-23, 1.61669130e-23,
       2.40008514e-23, 4.03532201e-23, 6.95747264e-23, 1.40957345e-22,
       2.84518575e-22, 5.17381239e-22, 1.08351297e-21, 2.15203017e-21,
       4.12016417e-21, 7.78952222e-21, 1.45615810e-20, 2.70914228e-20,
       4.94000000e-20])
atten_surv_ratio = np.zeros((n_cs,n_m))
signal_grid = np.zeros((n_cs,n_m))

In [3]:
result_dir = './result/'
result_path = Path(result_dir)
for filename in result_path.iterdir():
    file_stem = filename.stem
    # 正则表达式匹配 mDM 和 sigma 的值
    pattern = r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2"
    # 使用 re.search() 匹配并提取
    match = re.search(pattern, file_stem)
    mDM_in_GeV = float(match.group(1))  # 第一个括号组匹配 mDM 的值
    sigma_e_in_cm2 = float(match.group(2))  # 第二个括号组匹配 sigma 的值
    m_index = np.argmin(np.abs(m_grid - mDM_in_GeV))
    cs_index = n_cs - 1 - np.argmin(np.abs(cs_grid * center_line[m_index] - sigma_e_in_cm2))
    signal = np.loadtxt(filename, skiprows=2).item()
    signal_grid[cs_index, m_index] = signal

speed_pdf_dir = '../data/DM_Speed_PDF'
speed_pdf_path = Path(speed_pdf_dir)
for filename in speed_pdf_path.iterdir():
    if filename.suffix == '.txt':
        file_stem = filename.stem
        # 正则表达式匹配 mDM 和 sigma 的值
        pattern = r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2"
        # 使用 re.search() 匹配并提取
        match = re.search(pattern, file_stem)
        mDM_in_GeV = float(match.group(1))  # 第一个括号组匹配 mDM 的值
        sigma_e_in_cm2 = float(match.group(2))  # 第二个括号组匹配 sigma 的值
        m_index = np.argmin(np.abs(m_grid - mDM_in_GeV))
        cs_index = n_cs - 1 - np.argmin(np.abs(cs_grid * center_line[m_index] - sigma_e_in_cm2))
        speed_pdf = np.loadtxt(filename)
        atten_surv_ratio[cs_index, m_index] = scipy.integrate.simpson(speed_pdf[:,1], x = speed_pdf[:,0])

np.savetxt('./signal_grid.txt', signal_grid, fmt='%.3e')
np.savetxt('./atten_surv_ratio.txt', atten_surv_ratio, fmt='%.3e')